# 集群上lumpy运行流程

## 一、数据来源

### 197个样本路径：/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/

### 80组配对样本名称列表：/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt

## 二、配置环境与运行 

In [ ]:
conda create -n gridss_env r-base=4.3.2 -y
conda activate gridss_env

conda install -c bioconda -c conda-forge gridss hmftools-gripss bcftools samtools -y

In [ ]:
# 找安装在conda里面的 jar包
find $CONDA_PREFIX -name "gripss.jar"
# 找到路径为：/mnt/home/ygjx/chenkejin/anaconda3/envs/gridss_env/share/hmftools-gripss-2.4-0/gripss.jar

In [ ]:
# 看GRIPSS的版本及参数，显示版本为2.4
gripss -help

### 1、单样本处理

In [ ]:
# 上传作业
sbatch test_single_gripss.sh

### test_single_gripss.sh代码如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=Test_Single_GRIPSS
#SBATCH --nodes=1
#SBATCH --cpus-per-task=16                 # 测试同样分配 16 个 CPU
#SBATCH --mem=64G                          # 测试同样分配 64G 内存
#SBATCH --output=/mnt/home/ygjx/chenkejin/GRIDSS/logs/test_single_%j.log 

set -euo pipefail

# --- 0. 激活全新纯净环境 (已修正为 gripss_env) ---
source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate gridss_env

# --- 1. 全局变量与路径 ---
REF_FA="/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta"
BLACKLIST="/mnt/home/ygjx/chenkejin/GRIDSS/tracks/ENCFF356LFX.bed"
THREADS=16

# 【⚠️ 核心配置：已更新为 gridss_env 目录下的 jar 包路径】
# 如果你之前 find 出来的路径与这行不同，请以你 find 出来的为准替换掉这行！
GRIPSS_JAR="/mnt/home/ygjx/chenkejin/anaconda3/envs/gridss_env/share/hmftools-gripss-2.4-0/gripss.jar"

SOURCE_BASE="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
WORK_DIR="/mnt/home/ygjx/chenkejin/GRIDSS"

# --- 2. 设定测试样本 ---
PREFIX="1866277"
NORMAL_ID="${PREFIX}N"
TUMOR_ID="${PREFIX}T"

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] 单样本测试 ${PREFIX} 开始处理"
echo "全新环境: gripss_env | 16 CPUs, 64G Mem"
echo "=========================================================="

# --- 3. 目录构建 --
FINAL_VCF_DIR="${WORK_DIR}/Final_Results"
mkdir -p "$FINAL_VCF_DIR"

RUN_DIR="${WORK_DIR}/sandbox/${PREFIX}_gridss_test_run"
raw_vcf="${RUN_DIR}/${PREFIX}.gridss.raw.vcf"

# GRIPSS 默认以 Tumor ID 命名输出文件
somatic_vcf="${FINAL_VCF_DIR}/${TUMOR_ID}.gripss.vcf.gz"
final_unzipped_vcf="${FINAL_VCF_DIR}/${TUMOR_ID}.gripss.vcf"

# 强制清理以前残留的测试沙盒，保证本次测试绝对纯净
rm -rf "$RUN_DIR" && mkdir -p "$RUN_DIR"
cd "$RUN_DIR"

NORMAL_BAM="${SOURCE_BASE}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bam"
TUMOR_BAM="${SOURCE_BASE}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bam"

# --- 4. 核心流程 (GRIDSS -> GRIPSS -> 解压) ---
{
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 1/3: 运行 GRIDSS Joint Calling..."
    gridss \
        -r "$REF_FA" \
        -o "$raw_vcf" \
        -b "$BLACKLIST" \
        -t "$THREADS" \
        --workingdir "$RUN_DIR" \
        "$NORMAL_BAM" \
        "$TUMOR_BAM"

    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 2/3: 运行 Java 版 GRIPSS 过滤..."
    if [ -f "$raw_vcf" ]; then
        # 分配 32G 给 Java 虚拟机防止大样本内存溢出
        java -Xmx32G -jar "$GRIPSS_JAR" \
            -sample "$TUMOR_ID" \
            -reference "$NORMAL_ID" \
            -ref_genome_version 38 \
            -ref_genome "$REF_FA" \
            -vcf "$raw_vcf" \
            -output_dir "$FINAL_VCF_DIR" || { echo "⚠️ 致命错误：GRIPSS 运行失败！"; exit 1; }

        echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 3/3: 处理最终输出..."
        if [ -f "$somatic_vcf" ]; then
            gunzip -c "$somatic_vcf" > "$final_unzipped_vcf"
            echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] 单样本 ${PREFIX} 测试完成，成功生成解压版 VCF！"
        else
            echo "⚠️ 提示：样本 ${PREFIX} 经 GRIPSS 质控后无体细胞突变，创建空文件占位。"
            touch "$final_unzipped_vcf"
        fi
        
        # 测试成功后清理沙盒
        cd "${WORK_DIR}"
        rm -rf "$RUN_DIR"
    else
        echo "⚠️ 错误：未找到 $raw_vcf，GRIDSS 运行失败。"
        false
    fi

} || {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] 单样本 ${PREFIX} 运行中途失败！请检查日志。"
    exit 1
}

### tail -f logs/g_1866277_1611.log 实时查看日志

### 2、多样本批量处理

In [ ]:
# 上传作业
sbatch batch_gridss.sh

### batch_gridss.sh代码如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=GRIDSS_GRIPSS_Batch
#SBATCH --nodes=1
#SBATCH --cpus-per-task=16
#SBATCH --mem=64G
#SBATCH --array=1-80%20
#SBATCH --output=/mnt/home/ygjx/chenkejin/GRIDSS/logs/slurm_array_%A_%a.out 

set -euo pipefail

# --- 0. 激活正确的实际环境 ---
source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate gridss_env

# --- 1. 全局变量与路径 ---
REF_FA="/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta"
BLACKLIST="/mnt/home/ygjx/chenkejin/GRIDSS/tracks/ENCFF356LFX.bed"
THREADS=16
GRIPSS_JAR="/mnt/home/ygjx/chenkejin/anaconda3/envs/gridss_env/share/hmftools-gripss-2.4-0/gripss.jar"

SOURCE_BASE="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
WORK_DIR="/mnt/home/ygjx/chenkejin/GRIDSS"
TASK_LIST="/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt"

MASTER_LOG="${WORK_DIR}/master_progress_gripss.log"

# --- 2. 任务行解析 (无敌防爆版) ---
LINE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$TASK_LIST" | tr -d '\r' | xargs)

if [ -z "$LINE" ]; then
    exit 0
fi

NORMAL_ID=$(echo "$LINE" | awk '{print $1}')
TUMOR_ID=$(echo "$LINE" | awk '{print $2}')

if [ -z "$NORMAL_ID" ] || [ -z "$TUMOR_ID" ]; then
    exit 0
fi

if [[ "${NORMAL_ID,,}" == *"normal"* ]] || [[ "${NORMAL_ID,,}" == *"id"* ]]; then
    exit 0
fi

PREFIX=$(echo "$NORMAL_ID" | sed 's/N$//')
if [ -z "$PREFIX" ]; then
    exit 0
fi

# --- 3. 独立样本日志重定向 ---
mkdir -p "${WORK_DIR}/logs"
SAMPLE_LOG="${WORK_DIR}/logs/${PREFIX}.log"
exec > >(tee -i "$SAMPLE_LOG") 2>&1

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] 样本 ${PREFIX} 开始处理"
echo "全新引擎: GRIDSS + GRIPSS | 16 CPUs, 64G Mem"
echo "任务阵列 ID: ${SLURM_ARRAY_TASK_ID}/80  执行节点: $(hostname)"
echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX}" >> "$MASTER_LOG"

# --- 4. 目录构建与断点续传检测 ---
FINAL_VCF_DIR="${WORK_DIR}/Final_Results"
mkdir -p "$FINAL_VCF_DIR"

RUN_DIR="${WORK_DIR}/sandbox/${PREFIX}_gridss_run"
raw_vcf="${RUN_DIR}/${PREFIX}.gridss.raw.vcf"

somatic_vcf="${FINAL_VCF_DIR}/${TUMOR_ID}.gripss.vcf.gz"
final_unzipped_vcf="${FINAL_VCF_DIR}/${TUMOR_ID}.gripss.vcf"

if [ -f "$final_unzipped_vcf" ]; then
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] 样本 ${PREFIX} 已存在最终解压结果，跳过。"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] Sample ${PREFIX} skipped" >> "$MASTER_LOG"
    exit 0
fi

rm -rf "$RUN_DIR" && mkdir -p "$RUN_DIR"
cd "$RUN_DIR"

NORMAL_BAM="${SOURCE_BASE}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bam"
TUMOR_BAM="${SOURCE_BASE}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bam"

# --- 5. 核心流程执行 ---
{
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 1/3: 运行 GRIDSS Joint Calling..."
    gridss \
        -r "$REF_FA" \
        -o "$raw_vcf" \
        -b "$BLACKLIST" \
        -t "$THREADS" \
        --workingdir "$RUN_DIR" \
        "$NORMAL_BAM" \
        "$TUMOR_BAM"

    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 2/3: 运行 Java 版 GRIPSS 过滤..."
    if [ -f "$raw_vcf" ]; then
        java -Xmx32G -jar "$GRIPSS_JAR" \
            -sample "$TUMOR_ID" \
            -reference "$NORMAL_ID" \
            -ref_genome_version 38 \
            -ref_genome "$REF_FA" \
            -vcf "$raw_vcf" \
            -output_dir "$FINAL_VCF_DIR" || { echo "⚠️ 致命错误：GRIPSS 运行失败！"; exit 1; }

        echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 3/3: 处理最终输出..."
        if [ -f "$somatic_vcf" ]; then
            gunzip -c "$somatic_vcf" > "$final_unzipped_vcf"
            echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] 样本 ${PREFIX} 全流程处理并成功解压 VCF！"
        else
            echo "⚠️ 提示：样本 ${PREFIX} 经 GRIPSS 质控后无高置信度体细胞突变，创建空文件占位。"
            touch "$final_unzipped_vcf"
            echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] 样本 ${PREFIX} 流程闭环 (0 Variants)。"
        fi
        
        echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} completed" >> "$MASTER_LOG"
        
        cd "${WORK_DIR}"
        rm -rf "$RUN_DIR"
    else
        echo "⚠️ 错误：未找到 $raw_vcf，GRIDSS 第一阶段运行失败。"
        false
    fi

} || {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] 样本 ${PREFIX} 运行失败！请检查日志。"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} failed!" >> "$MASTER_LOG"
    exit 1
}


### tail -f master_progress_gripss.log 查看日志

### 结果路径：/mnt/home/ygjx/chenkejin/GRIDSS/Final_Results/